In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding, pipeline
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score
import os
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

In [24]:

# transformers.utils.logging.enable_progress_bar()
file_path = '/kaggle/input/wb-bad-words/train.csv'
data = pd.read_csv(file_path)
sorted_data = data.sort_values(by='label', ascending=False)
texts = data["text"][:50000].to_numpy()
labels = data["label"][:50000].to_numpy()
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels, 
    test_size=0.05, 
    stratify=labels, 
    random_state=42
)
train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels}).map(tokenize_function, batched=True)
val_dataset = Dataset.from_dict({"text": val_texts, "label": val_labels}).map(tokenize_function, batched=True)

class_weights = compute_class_weight('balanced', classes=[0, 1], y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# model_name = "DeepPavlov/rubert-base-cased"
# model_name = "sismetanin/rubert-toxic-pikabu-2ch"
model_name = "/kaggle/input/rubert_2ch/transformers/default/1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)





def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), axis=-1)
    f1 = f1_score(labels, predictions, average="weighted")
    acc = accuracy_score(labels, predictions)
    return {"f1": f1, "accuracy": acc}

training_args = TrainingArguments(
    output_dir = './results', 
    num_train_epochs = 3, 
    per_device_train_batch_size = 8, 
    per_device_eval_batch_size = 8, 
    weight_decay =0.01, 
    logging_dir = './logs', 
    load_best_model_at_end = True, 
    learning_rate = 1e-5,
    evaluation_strategy ='epoch',
    logging_strategy = 'epoch',
    save_strategy = 'epoch', 
    save_total_limit = 1,
    seed=21)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
class WeightedBCEWithLogitsLoss(nn.Module):
    def __init__(self, class_weights):
        super(WeightedBCEWithLogitsLoss, self).__init__()
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32)

    def forward(self, logits, targets):
        bce_loss = nn.BCEWithLogitsLoss(reduction='none')(logits, targets)
        weights = self.class_weights[targets.long()].view(-1)
        weighted_loss = (bce_loss * weights).mean()
        return weighted_loss




trainer.loss = WeightedBCEWithLogitsLoss(class_weights)



Map:   0%|          | 0/47500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_153/3042573504.py:69: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.class_weights = torch.tensor(class_weights, dtype=torch.float32)


In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.054400,0.045775,0.991966,0.992000
2,0.028900,0.065998,0.989540,0.989600
3,0.012800,0.065053,0.991570,0.991600


TrainOutput(global_step=17814, training_loss=0.03203442287562846, metrics={'train_runtime': 8530.159, 'train_samples_per_second': 16.705, 'train_steps_per_second': 2.088, 'total_flos': 3.74933253888e+16, 'train_loss': 0.03203442287562846, 'epoch': 3.0})

In [27]:
import shutil
shutil.make_archive("my_rubert_pretrained_2ch_2", 'zip', "/kaggle/working/results/checkpoint-5938")

from IPython.display import FileLink

FileLink('my_rubert_pretrained_2ch_2.zip')

/kaggle/working/my_rubert_pretrained_2ch_2.zip

In [28]:
def predict(model, tokenizer, data_path, save_path):
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
    predict_data = pd.read_csv(data_path)

    
    def truncate_text(text, max_words=512):
        words = text.split()
        if len(words) > max_words:
            return ' '.join(words[:max_words]), ' '.join(words[max_words:])
        else:
            return text, ''
    
    truncated_rows = []
    remaining_rows = []
    
    for index, row in predict_data.iterrows():
        truncated, remaining = truncate_text(row['text'])
        truncated_rows.append({'ID': row['ID'], 'text': truncated})
        if remaining:
            remaining_rows.append({'ID': row['ID'], 'text': remaining})
    
    df_truncated = pd.DataFrame(truncated_rows)
   

    predict_dataset = Dataset.from_dict({"text": df_truncated['text'].to_numpy()})
    predict_dataset = predict_dataset.map(tokenize_function, batched=True)
    tens_ids = torch.tensor(predict_dataset["input_ids"]).to("cuda")
    tens_attention_mask=torch.tensor(predict_dataset['attention_mask']).to("cuda")
    dataset = TensorDataset(tens_ids, tens_attention_mask)
    data_loader = DataLoader(dataset, batch_size=32)
    
    all_predictions = torch.tensor([]).to("cuda")
    model.eval() 
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="prediction", unit="batch"):

            batch_input_ids, batch_attention_mask = batch
    
            predictions = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask)
            all_predictions= torch.cat((all_predictions, predictions[0].argmax(axis=-1)))
        
    
    predict_data["label"] = (all_predictions.cpu()).numpy().astype(int)
    predict_data = predict_data.drop("text", axis=1)
    
    if len(remaining_rows) > 0:
        df_remaining = pd.DataFrame(remaining_rows)
        
        predict_dataset_remaining = Dataset.from_dict({"text": df_remaining['text'].to_numpy()})
        predict_dataset_remaining = predict_dataset.map(tokenize_function, batched=True)
        
        tens_ids_remaining = torch.tensor(predict_dataset_remaining["input_ids"]).to("cuda")
        tens_attention_mask_remaining=torch.tensor(predict_dataset_remaining['attention_mask']).to("cuda")
        
        dataset_remaining = TensorDataset(tens_ids, tens_attention_mask_remaining)
        data_loader_remaining = DataLoader(dataset_remaining, batch_size=32)
        
        all_predictions_remaning = torch.tensor([]).to("cuda")
            
        for batch in tqdm(data_loader, desc="prediction remaning", unit="batch"):
            batch_input_ids, batch_attention_mask = batch
            predictions = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask)
            all_predictions_remaning= torch.cat((all_predictions_remaning, predictions[0].argmax(axis=-1)))
            
        
        df_remaining["label"] = (all_predictions_remaning.cpu()).numpy().astype(int)
        df_remaining = df_remaining.drop("text", axis=1)
        for row in df_remaining.iterrows():
            if row["label"] == 0:
                continue
            else:
                predict_data.loc[predict_data['ID'] == row["ID"], 'label'] = 1
        
        del all_predictions_remaning
        del tens_attention_mask_remaning
        del tens_ids_remaning
            
    predict_data.to_csv(save_path, index=False)  
    print("saved file "+save_path)
    del all_predictions
    del tens_attention_mask
    del tens_ids
            

In [29]:

predict(model, tokenizer, '/kaggle/input/wb-bad-words/test.csv', "predict.csv")

Map:   0%|          | 0/60040 [00:00<?, ? examples/s]

prediction: 100%|██████████| 1877/1877 [15:31<00:00,  2.02batch/s]


saved file predict.csv
